In [1]:

from collections import defaultdict
import itertools
import nltk
from nltk import ngrams
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize
import numpy as np
import pandas as pd
import pickle
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Union, Optional
from torch.utils.data import DataLoader, Dataset

from dap_job_quality import config, PROJECT_DIR, logging
from dap_job_quality.getters.afs_data import get_eyp_ads, get_sim_occ_ads
from dap_job_quality.getters.models import sentence_classifier_pca, sentence_classifier_lr
from dap_job_quality.getters.data_getters import save_to_s3

# Load BERT model and tokenizer
model_name = config["sentence_model"]

nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

model = sentence_classifier_lr()
pca = sentence_classifier_pca()

MAX_LENGTH = 81 # 99th percentile of token length of sentences
SAMPLE_SIZE = 40000
SIMILARITY_THRESHOLD = 0

/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-06-17 19:44:43,104 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
class SentenceDataset(Dataset):
    def __init__(self, sentences, tokenizer, max_length=MAX_LENGTH):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return self.tokenizer(
            self.sentences[idx], padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )

# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

# Function to embed sentences
def embed_sentences(sentences: List[str], model_name: str, batch_size: int = 32, device: str = 'cuda' if torch.cuda.is_available() else 'cpu') -> torch.Tensor:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    
    dataset = SentenceDataset(sentences, tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    all_embeddings = []

    start_time = time.time()

    with torch.no_grad():
        for batch in dataloader:
            encoded_input = {key: val.squeeze().to(device) for key, val in batch.items()}
            model_output = model(**encoded_input)
            batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
            all_embeddings.append(batch_embeddings.cpu())

    elapsed_time = time.time() - start_time
    print(f"Batch size: {batch_size}, Time taken: {elapsed_time:.2f} seconds")

    return torch.cat(all_embeddings, dim=0)


def split_ngrams(text, length=6, n=4):
    if len(text.split()) > length:
        ngram_list = list(ngrams(text.split(), n))
    else:
        ngram_list = [text]
    return ngram_list

In [3]:
eyp = get_eyp_ads()
sim_occs = get_sim_occ_ads()
all_job_ads = pd.concat([eyp, sim_occs], axis=0).drop_duplicates()

lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v5.csv")

# Filter sentences which relate to job quality

In [4]:
logging.info(len(all_job_ads))
all_job_ads = all_job_ads[all_job_ads['created']>='2023-01-01']
logging.info(len(all_job_ads))
job_ads_sample = all_job_ads.sample(SAMPLE_SIZE, random_state=42)

2024-06-17 19:45:13,985 - root - INFO - 254529
2024-06-17 19:45:14,020 - root - INFO - 83150


In [5]:
job_ads_sample['sentences'] = job_ads_sample['clean_description'].apply(sent_tokenize)
job_ads_sample = job_ads_sample.explode('sentences')
logging.info(f'{len(job_ads_sample)} sentences')

2024-06-17 19:45:17,684 - root - INFO - 296782 sentences


In [12]:
def get_ngrams(text, n):
    text = text.split()
    texts_list = [
            lemmatizer.lemmatize(word)
            for word in text
            if word not in stop_words
        ]
    found_grams = ngrams(texts_list, n)
    
    return list(found_grams)

In [14]:
for n in range(2, 6):
    job_ads_sample[f'ngrams_{n}'] = job_ads_sample['sentences'].apply(lambda x: get_ngrams(x, n))
job_ads_sample.head()

,id,company_raw,job_title_raw,created,type,sector,parent_sector,knowledge_domain,occupation,min_annualised_salary,...,broad_ruc11,qualification_level,clean_job_title,matched_job_title,sentences,sentence_split,ngrams_2,ngrams_3,ngrams_4,ngrams_5
33648,50026592,Ceema Technology Recruitment Ltd,Nursery Room Leader,2023-03-16,Recruitment consultancy,Early Years Practitioner,Education,Education,Nursery Manager,22067.0,...,Predominantly Urban,None,NaN,NaN,nan,[],[],[],[],[]
181655,50699432,Reeson Education,Teaching Assistant,2023-06-26,Recruitment consultancy,Teaching Assistant,Education,Education,Teacher Assistant,20800.0,...,Predominantly Urban,NaN,teaching assistant,teaching assistant,Are you a recent graduate or an entry-level ca...,"[(Are, recent, graduate, entry-level), (recent...","[(Are, recent), (recent, graduate), (graduate,...","[(Are, recent, graduate), (recent, graduate, e...","[(Are, recent, graduate, entry-level), (recent...","[(Are, recent, graduate, entry-level, candidat..."
181655,50699432,Reeson Education,Teaching Assistant,2023-06-26,Recruitment consultancy,Teaching Assistant,Education,Education,Teacher Assistant,20800.0,...,Predominantly Urban,NaN,teaching assistant,teaching assistant,Do you have a passion for working with childre...,"[(Do, passion, working, child), (passion, work...","[(Do, passion), (passion, working), (working, ...","[(Do, passion, working), (passion, working, ch...","[(Do, passion, working, child), (passion, work...","[(Do, passion, working, child, special), (pass..."
181655,50699432,Reeson Education,Teaching Assistant,2023-06-26,Recruitment consultancy,Teaching Assistant,Education,Education,Teacher Assistant,20800.0,...,Predominantly Urban,NaN,teaching assistant,teaching assistant,"If so, we want to hear from you!","[(If, so,, want, hear), (so,, want, hear, you!)]","[(If, so,), (so,, want), (want, hear), (hear, ...","[(If, so,, want), (so,, want, hear), (want, he...","[(If, so,, want, hear), (so,, want, hear, you!)]","[(If, so,, want, hear, you!)]"
181655,50699432,Reeson Education,Teaching Assistant,2023-06-26,Recruitment consultancy,Teaching Assistant,Education,Education,Teacher Assistant,20800.0,...,Predominantly Urban,NaN,teaching assistant,teaching assistant,Here are some of the benefits of working with ...,"[(Here, benefit, working, u), (benefit, workin...","[(Here, benefit), (benefit, working), (working...","[(Here, benefit, working), (benefit, working, ...","[(Here, benefit, working, u), (benefit, workin...","[(Here, benefit, working, u, Competitive), (be..."


In [16]:
job_ads_sample_2 = job_ads_sample[['id', 'sentences', 'ngrams_2']].explode('ngrams_2')

In [17]:
job_ads_sample_2['ngrams_2'].value_counts()

ngrams_2
(Teaching, Assistant)    19832
(experience, working)     5361
(SEN, Teaching)           5254
(The, school)             4962
(teaching, assistant)     4239
                         ...  
(Claudia, submit)            1
(Title, PPA)                 1
(school, Stockwell.)         1
(lesson, experience)         1
(me.-5-Star, Google)         1
Name: count, Length: 396819, dtype: int64

In [ ]:
for n in range(2, 6):
    job_ads_sample[f'sentence_{n}grams'] = job_ads_sample['sentences'].apply(split_ngrams, n=n)

In [ ]:
def get_most_common_n_grams(
    texts_list: list, top_n: int = 100, ngram_range=range(2, 6)
) -> dict:
    """
    Get top_n most common ngrams from a list of texts
    """

    # Slightly clean and combine all texts in a list into a big string
    combined_texts = ""
    for topic_text in texts_list:
        combined_texts = combined_texts + " " + topic_text.lower()

    # Calculate most common ngrams for a few different n
    results = {}
    for n in ngram_range:
        combined_texts_list = combined_texts.split()
        combined_texts_list = [
            lemmatizer.lemmatize(word)
            for word in combined_texts_list
            if word not in stop_words
        ]
        found_grams = ngrams(combined_texts_list, n)
        all_grams = []
        for grams in found_grams:
            all_grams.append(" ".join(grams))
        results[n] = [
            (a, b, round(b / len(all_grams), 6))
            for a, b in Counter(all_grams).most_common(top_n)
        ]
    return results

In [ ]:
ad_embeddings = embed_sentences(job_ads_sample['sentences'].tolist(), model_name, 64)

In [ ]:
X_new_pca = pca.transform(ad_embeddings)

predictions = model.predict(X_new_pca)

In [ ]:
job_ads_sample['job_quality'] = predictions

In [ ]:
jq_sentences_df = job_ads_sample[job_ads_sample['job_quality']==1]

# Get ngrams from JQ sentences

In [ ]:
jq_sentences_df['ngrams'] = jq_sentences_df['sentences'].apply(lambda x: split_ngrams(x, 6, 4))
jq_sentences_df_long = jq_sentences_df.explode('ngrams')
jq_sentences_df_long['ngrams'] = jq_sentences_df_long['ngrams'].apply(lambda x: ' '.join(x) if isinstance(x, tuple) else x)
jq_sentences_df_long.head()

# Calculate cosine similarity of unique ngrams to target phrases

In [ ]:
unique_ngrams = list(jq_sentences_df_long['ngrams'].unique())

In [ ]:
target_phrases = lookup['target_phrase'].tolist()

In [ ]:
# Embed the target phrases
target_embeddings = embed_sentences(target_phrases)
    
ngram_embeddings = embed_sentences(unique_ngrams)
        
similarities = cosine_similarity(ngram_embeddings, target_embeddings)

similarities
        
matches = defaultdict(list)
for i, ngram in enumerate(unique_ngrams):
    for j, target_phrase in enumerate(target_phrases):
        if similarities[i, j] > SIMILARITY_THRESHOLD:
            matches[ngram].append((target_phrase, similarities[i, j]))
            
# Deduplicate target phrases for each text
deduplicated_matches = {}
for ngram, matches_list in matches.items():
    unique_matches = list({phrase: sim for phrase, sim in matches_list}.items())
    deduplicated_matches[ngram] = unique_matches

In [ ]:
matches_df = pd.DataFrame(deduplicated_matches.items(), columns=['ngram', 'matches'])

In [ ]:
jq_sentences_df_long = pd.merge(jq_sentences_df_long, matches_df, how='left', left_on='ngrams', right_on='ngram')
jq_sentences_df_long.head()

In [ ]:
save_to_s3(BUCKET_NAME, jq_sentences_df_long, f'job_quality/early_years/jq_sentences/jq_sentences_2023_out_of_{SAMPLE_SIZE}.csv')